In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, f1_score, average_precision_score
from pathlib import Path
import optuna
from optuna.visualization import plot_optimization_history, plot_param_importances
import warnings
warnings.filterwarnings('ignore')

# %% Configuration
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
OUTPUT_DIR = Path("../outputs")
MODEL_DIR = Path("../models")
PLOT_DIR = Path("../plots")
MODEL_DIR.mkdir(exist_ok=True)
PLOT_DIR.mkdir(exist_ok=True)

print(f"Using device: {DEVICE}")

# %% Dataset
class ICUBinaryDataset(Dataset):
    def __init__(self, sequences, static, labels):
        self.sequences = torch.tensor(sequences, dtype=torch.float32)
        self.static = torch.tensor(static, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.float32)
    
    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):
        return {
            'sequence': self.sequences[idx], 
            'static': self.static[idx],     
            'label': self.labels[idx]       
        }

# %% Model Architecture
class ICURNNClassifier(nn.Module):
    def __init__(self, n_temporal, n_static, hidden_dim, n_layers, rnn_type, dropout, bidirectional):
        super().__init__()
        
        if rnn_type == 'LSTM':
            self.rnn = nn.LSTM(
                n_temporal, hidden_dim, n_layers, 
                batch_first=True, 
                dropout=dropout if n_layers > 1 else 0,
                bidirectional=bidirectional
            )
        else:
            self.rnn = nn.GRU(
                n_temporal, hidden_dim, n_layers, 
                batch_first=True, 
                dropout=dropout if n_layers > 1 else 0,
                bidirectional=bidirectional
            )
        
        rnn_output_dim = hidden_dim * (2 if bidirectional else 1)
        
        # Static network
        self.static_net = nn.Sequential(
            nn.Linear(n_static, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout)
        )
        
        # Classifier
        combined_dim = rnn_output_dim + (hidden_dim // 2)
        self.classifier = nn.Sequential(
            nn.Linear(combined_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, 1)
        )
        
        self.bidirectional = bidirectional

    def forward(self, x_seq, x_static):
        rnn_out, h_n = self.rnn(x_seq)
        
        if isinstance(h_n, tuple):
            h_n = h_n[0]
        
        if self.bidirectional:
            last_h = torch.cat([h_n[-2], h_n[-1]], dim=1)
        else:
            last_h = h_n[-1]
        
        static_h = self.static_net(x_static)
        combined = torch.cat([last_h, static_h], dim=1)
        return self.classifier(combined).squeeze(-1)

# %% Training Function for Optuna
def train_with_hyperparams(trial, rnn_type, train_loader, val_loader, pos_weight, n_temporal, n_static):
    # Hyperparameters to tune
    hidden_dim = trial.suggest_categorical('hidden_dim', [64, 96, 128, 160, 192])
    n_layers = trial.suggest_int('n_layers', 2, 4)
    dropout = trial.suggest_float('dropout', 0.2, 0.5)
    learning_rate = trial.suggest_float('lr', 1e-4, 1e-2, log=True)
    weight_decay = trial.suggest_float('weight_decay', 1e-5, 1e-3, log=True)
    bidirectional = trial.suggest_categorical('bidirectional', [True, False])
    pos_weight_multiplier = trial.suggest_float('pos_weight_mult', 1.0, 2.0)
    batch_norm = trial.suggest_categorical('batch_norm', [True, False])
    
    # Create model
    model = ICURNNClassifier(
        n_temporal, n_static, hidden_dim, n_layers, 
        rnn_type, dropout, bidirectional
    ).to(DEVICE)
    
    # Optimizer
    optimizer = optim.AdamW(
        model.parameters(), 
        lr=learning_rate, 
        weight_decay=weight_decay
    )
    
    # Scheduler
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=5, min_lr=1e-6
    )
    
    # Loss with adjusted pos_weight
    adjusted_pos_weight = pos_weight * pos_weight_multiplier
    criterion = nn.BCEWithLogitsLoss(pos_weight=adjusted_pos_weight.to(DEVICE))
    
    # Training loop (reduced epochs for faster tuning)
    n_epochs = 30
    best_val_auroc = 0.0
    patience = 10
    patience_counter = 0
    
    for epoch in range(n_epochs):
        # Training
        model.train()
        train_loss = 0
        for batch in train_loader:
            optimizer.zero_grad()
            logits = model(batch['sequence'].to(DEVICE), batch['static'].to(DEVICE))
            loss = criterion(logits, batch['label'].to(DEVICE))
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            train_loss += loss.item()
        
        # Validation
        model.eval()
        val_loss = 0
        all_probs = []
        all_labels = []
        
        with torch.no_grad():
            for batch in val_loader:
                logits = model(batch['sequence'].to(DEVICE), batch['static'].to(DEVICE))
                loss = criterion(logits, batch['label'].to(DEVICE))
                val_loss += loss.item()
                
                probs = torch.sigmoid(logits)
                all_probs.extend(probs.cpu().numpy())
                all_labels.extend(batch['label'].numpy())
        
        avg_val = val_loss / len(val_loader)
        scheduler.step(avg_val)
        
        # Calculate metrics
        val_auroc = roc_auc_score(all_labels, all_probs)
        val_auprc = average_precision_score(all_labels, all_probs)
        
        # Combined metric (weighted average)
        combined_metric = 0.6 * val_auroc + 0.4 * val_auprc
        
        # Early stopping
        if combined_metric > best_val_auroc:
            best_val_auroc = combined_metric
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                break
        
        # Report to Optuna
        trial.report(combined_metric, epoch)
        
        # Prune unpromising trials
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()
    
    return best_val_auroc

# %% Load Data
print("\n" + "="*60)
print("LOADING DATA")
print("="*60)

data = np.load(OUTPUT_DIR / 'dl_sequences_all.npz')
N_TEMP = data['X_train_seq'].shape[-1]
N_STAT = data['X_train_static'].shape[-1]

print(f"Temporal features: {N_TEMP}")
print(f"Static features: {N_STAT}")

# Calculate class weights
pos_count = float(data['y_train_binary'].sum())
neg_count = float(len(data['y_train_binary']) - data['y_train_binary'].sum())
pos_weight = torch.tensor([neg_count / pos_count], dtype=torch.float32)

print(f"\nClass balance:")
print(f"  Negative: {int(neg_count)} ({neg_count/(neg_count+pos_count)*100:.1f}%)")
print(f"  Positive: {int(pos_count)} ({pos_count/(neg_count+pos_count)*100:.1f}%)")
print(f"  Base pos_weight: {pos_weight.item():.3f}")

# Create dataloaders
train_loader = DataLoader(
    ICUBinaryDataset(data['X_train_seq'], data['X_train_static'], data['y_train_binary']), 
    batch_size=64, shuffle=True
)
val_loader = DataLoader(
    ICUBinaryDataset(data['X_val_seq'], data['X_val_static'], data['y_val_binary']), 
    batch_size=64
)

# %% Hyperparameter Tuning
print("\n" + "="*60)
print("HYPERPARAMETER TUNING")
print("="*60)

for rnn_type in ['LSTM', 'GRU']:
    print(f"\n{'='*60}")
    print(f"Tuning {rnn_type}")
    print(f"{'='*60}")
    
    # Create Optuna study
    study = optuna.create_study(
        direction='maximize',
        pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=10)
    )
    
    # Optimize
    study.optimize(
        lambda trial: train_with_hyperparams(
            trial, rnn_type, train_loader, val_loader, pos_weight, N_TEMP, N_STAT
        ),
        n_trials=50,  # Increase for better results (but takes longer)
        timeout=None,
        show_progress_bar=True
    )
    
    # Best parameters
    print(f"\n{rnn_type} - Best trial:")
    print(f"  Value (Combined Metric): {study.best_trial.value:.4f}")
    print(f"  Params:")
    for key, value in study.best_trial.params.items():
        print(f"    {key}: {value}")
    
    # Save best parameters
    best_params = study.best_trial.params
    best_params['rnn_type'] = rnn_type
    best_params['combined_metric'] = study.best_trial.value
    
    with open(MODEL_DIR / f'best_hyperparams_{rnn_type.lower()}.txt', 'w') as f:
        for key, value in best_params.items():
            f.write(f"{key}: {value}\n")
    
    # Visualizations
    try:
        # Optimization history
        fig1 = plot_optimization_history(study)
        fig1.write_image(PLOT_DIR / f'{rnn_type.lower()}_optimization_history.png')
        
        # Parameter importances
        fig2 = plot_param_importances(study)
        fig2.write_image(PLOT_DIR / f'{rnn_type.lower()}_param_importance.png')
        
        print(f"\n  Visualizations saved to {PLOT_DIR}")
    except Exception as e:
        print(f"  Could not create visualizations: {e}")

# %% Train Final Models with Best Hyperparameters
print("\n" + "="*60)
print("TRAINING FINAL MODELS WITH BEST HYPERPARAMETERS")
print("="*60)

# Load test data
test_loader = DataLoader(
    ICUBinaryDataset(data['X_test_seq'], data['X_test_static'], data['y_test_binary']), 
    batch_size=64
)

final_results = {}

for rnn_type in ['LSTM', 'GRU']:
    print(f"\n{'='*60}")
    print(f"Training Final {rnn_type} Model")
    print(f"{'='*60}")
    
    # Load best hyperparameters
    best_params = {}
    with open(MODEL_DIR / f'best_hyperparams_{rnn_type.lower()}.txt', 'r') as f:
        for line in f:
            key, value = line.strip().split(': ')
            if key in ['hidden_dim', 'n_layers']:
                best_params[key] = int(value)
            elif key in ['dropout', 'lr', 'weight_decay', 'pos_weight_mult']:
                best_params[key] = float(value)
            elif key in ['bidirectional', 'batch_norm']:
                best_params[key] = value == 'True'
    
    print("Best hyperparameters:")
    for key, value in best_params.items():
        print(f"  {key}: {value}")
    
    # Create model
    model = ICURNNClassifier(
        N_TEMP, N_STAT, 
        best_params['hidden_dim'], 
        best_params['n_layers'],
        rnn_type,
        best_params['dropout'],
        best_params['bidirectional']
    ).to(DEVICE)
    
    # Optimizer
    optimizer = optim.AdamW(
        model.parameters(), 
        lr=best_params['lr'], 
        weight_decay=best_params['weight_decay']
    )
    
    # Scheduler
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=7, min_lr=1e-6
    )
    
    # Loss
    adjusted_pos_weight = pos_weight * best_params['pos_weight_mult']
    criterion = nn.BCEWithLogitsLoss(pos_weight=adjusted_pos_weight.to(DEVICE))
    
    # Full training (more epochs)
    best_val_auroc = 0.0
    patience = 20
    patience_counter = 0
    
    for epoch in range(100):
        # Training
        model.train()
        train_loss = 0
        for batch in train_loader:
            optimizer.zero_grad()
            logits = model(batch['sequence'].to(DEVICE), batch['static'].to(DEVICE))
            loss = criterion(logits, batch['label'].to(DEVICE))
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            train_loss += loss.item()
        
        # Validation
        model.eval()
        val_loss = 0
        all_probs = []
        all_labels = []
        
        with torch.no_grad():
            for batch in val_loader:
                logits = model(batch['sequence'].to(DEVICE), batch['static'].to(DEVICE))
                loss = criterion(logits, batch['label'].to(DEVICE))
                val_loss += loss.item()
                
                probs = torch.sigmoid(logits)
                all_probs.extend(probs.cpu().numpy())
                all_labels.extend(batch['label'].numpy())
        
        avg_train = train_loss / len(train_loader)
        avg_val = val_loss / len(val_loader)
        scheduler.step(avg_val)
        
        val_auroc = roc_auc_score(all_labels, all_probs)
        val_auprc = average_precision_score(all_labels, all_probs)
        
        print(f"Epoch {epoch+1}: Train Loss {avg_train:.4f}, Val Loss {avg_val:.4f}, "
              f"AUROC {val_auroc:.4f}, AUPRC {val_auprc:.4f}")
        
        # Save best model
        if val_auroc > best_val_auroc:
            best_val_auroc = val_auroc
            torch.save(model.state_dict(), MODEL_DIR / f'tuned_best_{rnn_type.lower()}.pt')
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"Early stopping at epoch {epoch+1}")
                break
    
    # Test evaluation
    model.load_state_dict(torch.load(MODEL_DIR / f'tuned_best_{rnn_type.lower()}.pt'))
    model.eval()
    
    y_probs = []
    y_true = []
    
    with torch.no_grad():
        for batch in test_loader:
            logits = model(batch['sequence'].to(DEVICE), batch['static'].to(DEVICE))
            probs = torch.sigmoid(logits)
            y_probs.extend(probs.cpu().numpy())
            y_true.extend(batch['label'].numpy())
    
    # Optimize threshold
    val_probs = []
    val_labels = []
    with torch.no_grad():
        for batch in val_loader:
            logits = model(batch['sequence'].to(DEVICE), batch['static'].to(DEVICE))
            probs = torch.sigmoid(logits)
            val_probs.extend(probs.cpu().numpy())
            val_labels.extend(batch['label'].numpy())
    
    best_f1 = 0
    best_threshold = 0.5
    for threshold in np.arange(0.3, 0.7, 0.01):
        val_pred = [1 if p > threshold else 0 for p in val_probs]
        current_f1 = f1_score(val_labels, val_pred)
        if current_f1 > best_f1:
            best_f1 = current_f1
            best_threshold = threshold
    
    y_pred = [1 if p > best_threshold else 0 for p in y_probs]
    
    test_auroc = roc_auc_score(y_true, y_probs)
    test_auprc = average_precision_score(y_true, y_probs)
    test_f1 = f1_score(y_true, y_pred)
    
    final_results[rnn_type] = {
        'auroc': test_auroc,
        'auprc': test_auprc,
        'f1': test_f1,
        'threshold': best_threshold
    }
    
    print(f"\n{rnn_type} Final Test Results (Threshold: {best_threshold:.3f}):")
    print(f"  ROC-AUC: {test_auroc:.4f}")
    print(f"  AUPRC: {test_auprc:.4f}")
    print(f"  F1-Score: {test_f1:.4f}")

# %% Summary
print("\n" + "="*60)
print("HYPERPARAMETER TUNING COMPLETE")
print("="*60)

summary_df = pd.DataFrame(final_results).T
summary_df.to_csv(PLOT_DIR / 'tuned_results_summary.csv')

print("\nFinal Results Summary:")
print(summary_df)
print(f"\nSaved to: {PLOT_DIR / 'tuned_results_summary.csv'}")

/Users/suki/mamba/envs/main_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[I 2026-01-03 21:51:04,637] A new study created in memory with name: no-name-83aec85f-cdab-4ee0-9a52-a955693f58bb


Using device: cpu

LOADING DATA
Temporal features: 34
Static features: 10

Class balance:
  Negative: 16453 (78.7%)
  Positive: 4465 (21.3%)
  Base pos_weight: 3.685

HYPERPARAMETER TUNING

Tuning LSTM


Best trial: 0. Best value: 0.763387:   2%|▏         | 1/50 [07:57<6:29:47, 477.29s/it]

[I 2026-01-03 21:59:01,936] Trial 0 finished with value: 0.7633869649730817 and parameters: {'hidden_dim': 128, 'n_layers': 3, 'dropout': 0.4315650505866214, 'lr': 0.006395631813581068, 'weight_decay': 0.00010455238166996444, 'bidirectional': True, 'pos_weight_mult': 1.91355218747987, 'batch_norm': True}. Best is trial 0 with value: 0.7633869649730817.


Best trial: 0. Best value: 0.763387:   4%|▍         | 2/50 [11:29<4:16:54, 321.13s/it]

...

[I 2026-01-04 05:49:58,684] Trial 47 finished with value: 0.766415106229797 and parameters: {'hidden_dim': 192, 'n_layers': 3, 'dropout': 0.30159673169223433, 'lr': 0.0037547867598031323, 'weight_decay': 2.243849416023914e-05, 'bidirectional': True, 'pos_weight_mult': 1.1574939878793602, 'batch_norm': True}. Best is trial 22 with value: 0.7710950824302774.


Best trial: 22. Best value: 0.771095:  98%|█████████▊| 49/50 [4:12:22<07:06, 426.25s/it]

[I 2026-01-04 05:59:18,638] Trial 48 finished with value: 0.7657558954981231 and parameters: {'hidden_dim': 192, 'n_layers': 3, 'dropout': 0.3600630979677336, 'lr': 0.0019638946269076364, 'weight_decay': 1.3265423614555376e-05, 'bidirectional': True, 'pos_weight_mult': 1.3383573862523028, 'batch_norm': False}. Best is trial 22 with value: 0.7710950824302774.


Best trial: 22. Best value: 0.771095: 100%|██████████| 50/50 [4:14:27<00:00, 305.36s/it]


[I 2026-01-04 06:01:24,222] Trial 49 finished with value: 0.7609671843818382 and parameters: {'hidden_dim': 128, 'n_layers': 3, 'dropout': 0.2508443050933126, 'lr': 0.0009612053164033291, 'weight_decay': 3.273597739454414e-05, 'bidirectional': False, 'pos_weight_mult': 1.598601682988037, 'batch_norm': True}. Best is trial 22 with value: 0.7710950824302774.

GRU - Best trial:
  Value (Combined Metric): 0.7711
  Params:
    hidden_dim: 160
    n_layers: 3
    dropout: 0.28644182401748813
    lr: 0.004299697121312118
    weight_decay: 0.00012089088544831119
    bidirectional: True
    pos_weight_mult: 1.8487688658848385
    batch_norm: True
  Could not create visualizations: Tried to import 'plotly' but failed. Please make sure that the package is installed correctly to use this feature. Actual error: No module named 'plotly'.

TRAINING FINAL MODELS WITH BEST HYPERPARAMETERS

Training Final LSTM Model
Best hyperparameters:
  hidden_dim: 192
  n_layers: 2
  dropout: 0.200994973733184
  lr: